In [48]:
import torch 
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import re,time
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import nltk
import os
import helper as h
from torchtext.data.utils import get_tokenizer
import spacy
from collections import Counter
from torchtext.vocab import build_vocab_from_iterator
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pad_sequence

['.ipynb_checkpoints',
 'Dataset',
 'helper.py',
 'mode.py',
 'Sentiment Analysis .ipynb',
 '__pycache__']

In [6]:
data=pd.read_csv('Dataset/IMDB Dataset.csv')

In [7]:
data['review']=data['review'].apply(h.remove_html_tags)

In [8]:
data['review']=data['review'].apply(h.remove_urls)

In [9]:
data['review']=data['review'].apply(h.repalce_chatwords)

In [10]:
data.isnull().sum()

review       0
sentiment    0
dtype: int64

In [11]:
data['sentiment'].unique()

<ArrowStringArray>
['positive', 'negative']
Length: 2, dtype: str

In [12]:
data['sentiment']=data['sentiment'].map({'positive':1, 'negative':0})

In [13]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. The filming tec...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [14]:
data.iloc[0]

review       One of the other reviewers has mentioned that ...
sentiment                                                    1
Name: 0, dtype: object

In [15]:
def lower_case(text):
    return text.lower()

In [16]:
data['review']=data['review'].apply(lower_case)

In [17]:
data.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production. the filming tec...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there's a family where a little boy ...,0
4,"petter mattei's ""love in the time of money"" is...",1


In [18]:
nlp = spacy.load("en_core_web_sm")

In [19]:
tokenizer=get_tokenizer('spacy',language='en_core_web_sm')

In [20]:
text = "Hello, welcome to Natural Language Processing!"

tokens = tokenizer(text)

print(tokens)

['Hello', ',', 'welcome', 'to', 'Natural', 'Language', 'Processing', '!']


In [21]:
data['token']=data['review'].apply(tokenizer)

In [22]:
data.head()

,review,sentiment,token
0,one of the other reviewers has mentioned that ...,1,"[one, of, the, other, reviewers, has, mentione..."
1,a wonderful little production. the filming tec...,1,"[a, wonderful, little, production, ., the, fil..."
2,i thought this was a wonderful way to spend ti...,1,"[i, thought, this, was, a, wonderful, way, to,..."
3,basically there's a family where a little boy ...,0,"[basically, there, 's, a, family, where, a, li..."
4,"petter mattei's ""love in the time of money"" is...",1,"[petter, mattei, 's, "", love, in, the, time, o..."


In [23]:
train_data,test_data=train_test_split(data,test_size=0.2,random_state=42)

In [24]:
word_counts = Counter()

for tokens in train_data["token"]:
    if isinstance(tokens, list):
        word_counts.update(tokens)

print(word_counts.most_common(10))

[('the', 533272), ('.', 436724), (',', 433689), ('and', 258390), ('a', 256663), ('of', 231545), ('to', 214975), ('is', 173683), ('it', 151786), ('in', 148764)]


In [25]:
vocab_size=20000
word2idx = {
    "<PAD>": 0,
    "<UNK>": 1
}
for word, count in word_counts.most_common(vocab_size - 2):
    word2idx[word] = len(word2idx)
print("Vocabulary size:", len(word2idx))

Vocabulary size: 20000


In [26]:
train_data.head()

,review,sentiment,token
39087,that's what i kept asking myself during the ma...,0,"[that, 's, what, i, kept, asking, myself, duri..."
30893,i did not watch the entire movie. i could not ...,0,"[i, did, not, watch, the, entire, movie, ., i,..."
45278,a touching love story reminiscent of in the m...,1,"[a, touching, love, story, reminiscent, of, i..."
16398,this latter-day fulci schlocker is a totally a...,0,"[this, latter, -, day, fulci, schlocker, is, a..."
13653,"first of all, i firmly believe that norwegian ...",0,"[first, of, all, ,, i, firmly, believe, that, ..."


In [27]:
def numericalize(tokens):
    return [
        word2idx.get(word, word2idx["<UNK>"])
        for word in tokens
    ]

train_data["input_ids"] = train_data["token"].apply(numericalize)
test_data["input_ids"] = test_data["token"].apply(numericalize)

In [28]:
print(train_data[["token", "input_ids"]].head())

                                                   token  \
39087  [that, 's, what, i, kept, asking, myself, duri...   
30893  [i, did, not, watch, the, entire, movie, ., i,...   
45278  [a, touching, love, story, reminiscent, of, i...   
16398  [this, latter, -, day, fulci, schlocker, is, a...   
13653  [first, of, all, ,, i, firmly, believe, that, ...   

                                               input_ids  
39087  [14, 16, 57, 12, 800, 2155, 544, 316, 2, 121, ...  
30893  [12, 81, 30, 116, 2, 453, 21, 3, 12, 88, 30, 1...  
45278  [6, 1336, 126, 74, 2841, 7, 1, 2, 1276, 20, 12...  
16398  [13, 1578, 17, 263, 2696, 1, 9, 6, 458, 4483, ...  
13653  [101, 7, 38, 4, 12, 5491, 270, 14, 8214, 110, ...  


In [29]:
print(type(train_data["token"].iloc[0]))
print(train_data["token"].iloc[0])

<class 'list'>
['that', "'s", 'what', 'i', 'kept', 'asking', 'myself', 'during', 'the', 'many', 'fights', ',', 'screaming', 'matches', ',', 'swearing', 'and', 'general', 'mayhem', 'that', 'permeate', 'the', '84', 'minutes', '.', 'the', 'comparisons', 'also', 'stand', 'up', 'when', 'you', 'think', 'of', 'the', 'one', '-', 'dimensional', 'characters', ',', 'who', 'have', 'so', 'little', 'depth', 'that', 'it', 'is', 'virtually', 'impossible', 'to', 'care', 'what', 'happens', 'to', 'them', '.', 'they', 'are', 'just', 'badly', 'written', 'cyphers', 'for', 'the', 'director', 'to', 'hang', 'his', 'multicultural', 'beliefs', 'on', ',', 'a', 'topic', 'that', 'has', 'been', 'done', 'much', 'better', 'in', 'other', 'dramas', 'both', 'on', 'tv', 'and', 'the', 'cinema', '.', 'i', 'must', 'confess', ',', 'i', "'m", 'not', 'really', 'one', 'for', 'spotting', 'bad', 'performances', 'during', 'a', 'film', ',', 'but', 'it', 'must', 'be', 'said', 'that', 'nichola', 'burley', '(', 'as', 'the', 'heroine', 

In [30]:
class IMDBDataset(Dataset):
    def __init__(self,data):
        self.reviews=data['input_ids'].tolist()
        self.label=data['sentiment'].tolist()

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self,idx):
        return (torch.tensor(self.reviews[idx],dtype=torch.long),
                torch.tensor(self.label[idx],dtype=torch.float)
                )

In [31]:
PAD_IDX = word2idx["<PAD>"]
def collate_fn(batch):
    reviews,labels=zip(*batch)
    lenghts=torch.tensor([len(x) for x in reviews])
    reviews=pad_sequence(reviews,batch_first=True,padding_value=PAD_IDX)
    labels=torch.stack(labels)
    return reviews,lenghts,labels

In [32]:
train_dataset=IMDBDataset(train_data)
test_dataset=IMDBDataset(test_data)
train_loader=DataLoader(train_dataset,
                        batch_size=64,
                        shuffle=True,
                        collate_fn=collate_fn
                       )
test_loader=DataLoader(test_dataset,
                       batch_size=64,
                       shuffle=False,
                       collate_fn=collate_fn
                      )

In [33]:
reviews, lengths, labels = next(iter(train_loader))

print("Reviews:", reviews.shape)
print("Lengths:", lengths.shape)
print("Labels:", labels.shape)

Reviews: torch.Size([64, 1634])
Lengths: torch.Size([64])
Labels: torch.Size([64])


In [44]:
class RNN(nn.Module):
    def __init__(self,vocab_size,embeding_dim,hid_dim):
        super(RNN,self).__init__()
        self.embedding=nn.Embedding(num_embeddings=vocab_size,
                                    embedding_dim=embedding_dim,
                                    )
        self.rnn=nn.RNN(input_size=embedding_dim,
                        hidden_size=hid_dim,
                        batch_first=True)
        self.fc=nn.Linear(in_features=hid_dim,out_features=1)
        self.dropout = nn.Dropout(0.4)
    def forward(self,x,length):
        embedded=self.embedding(x)
        from torch.nn.utils.rnn import pack_padded_sequence
        packed=pack_padded_sequence(embedded,
                                  length.cpu(),
                                  batch_first=True,
                                  enforce_sorted=False)
        out,hidden=self.rnn(packed)
        hidden=hidden[-1]
        hidden=self.dropout(hidden)
        result=self.fc(hidden)
        return result.squeeze(1)


In [45]:
vocab_size=len(word2idx)
embedding_dim=100
hidden_dim=128

device='cuda' if torch.cuda.is_available() else 'cpu'

model=RNN(vocab_size=vocab_size,
          embeding_dim=embedding_dim,
          hid_dim=hidden_dim
         ).to(device)
print(model)

RNN(
  (embedding): Embedding(20000, 100)
  (rnn): RNN(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (dropout): Dropout(p=0.4, inplace=False)
)


In [43]:
optimizer=optim.Adam(model.parameters(),lr=0.001)
loss_fn=nn.BCEWithLogitsLoss()

In [ ]:
epochs=11
total_train_loss=[]
for epoch in range(epochs):
    model.train()
    total_loss=0.0
    correct=0
    total=0
    for reviews,length,labels in train_loader:
        reviews=reviews.to(device)
        labels=labels.to(device)
        optimizer.zero_grad()
        pred=model(reviews,length)
        loss=loss_fn(pred,labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1)
        optimizer.step()

        total_loss+=loss.item()
        prediction=(torch.sigmoid(pred)>=0.5).float()
        correct+=(prediction==labels).sum().item()
        total+=labels.size(0)
    epoch_loss=total_loss/len(train_loader)
    total_train_loss.append(epoch_loss)
    accuray=(correct/total) *100

    # if epoch %10==0:
    print(f"Epoch :{epoch}    |  Epoch Loss :{epoch_loss}    | Accuracy  :{accuray} %")
    

In [ ]:
epochs = 11
total_train_loss = []

for epoch in range(epochs):

    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for reviews, length, labels in train_loader:

        reviews = reviews.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        pred = model(reviews, length)

        loss = loss_fn(pred, labels.float())

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1
        )

        optimizer.step()

        total_loss += loss.item()

        prediction = (torch.sigmoid(pred) >= 0.5).float()

        correct += (prediction == labels).sum().item()
        total += labels.numel()

    epoch_loss = total_loss / len(train_loader)

    total_train_loss.append(epoch_loss)

    accuracy = (correct / total) * 100

    print(
        f"Epoch: {epoch+1} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {accuracy:.2f}%"
    )

In [46]:
model = RNN(
    vocab_size=vocab_size,
    embeding_dim=embedding_dim,
    hid_dim=hidden_dim
)

In [47]:
checkpoint=torch.load(
    'model/imdb_rnn_model.pth',
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
model.to(device)
wrod2idx=checkpoint['word2idx']
max_length=checkpoint['max_length']
print("full model loaded")


full model loaded


In [49]:
def process_review(model,text):
    text=h.remove_html_tags(text)
    text=h.remove_urls(text)
    text=h.check_spelling(text)
    text=h.repalce_chatwords(text)
    text=h.demojize_text(text)
    tokens=tokenizer(text)
    ids = [
        word2idx.get(token, word2idx['<UNK>'])
        for token in tokens
    ]
    ids = ids[:max_length]
    length = len(ids)
    ids += [word2idx['<PAD>']] * (max_length - len(ids))
    review = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        pred = model(review, length=torch.tensor([length]))
        probability = torch.sigmoid(pred).item()
    sentiment = 'Positive' if probability >= 0.5 else 'Negative'

    return sentiment, probability
    
    

In [50]:
text = "This movie was absolutely amazing. I loved it!"

sentiment, probability = process_review(model, text)

print("Sentiment:", sentiment)
print("Probability:", probability)

Sentiment: Positive
Probability: 0.9127485156059265


In [52]:
text1 = "The movie started with an interesting concept and the first thirty minutes were genuinely entertaining. However, as the story progressed, everything became predictable. The actors did a decent job, especially the lead actress, but the dialogue felt unnatural and several scenes seemed unnecessarily long. I wouldn't call it a terrible movie, but I certainly wouldn't recommend watching it twice. It was okay for a lazy Sunday afternoon."

sentiment1, probability1 = process_review(model, text1)

print("Sentiment:", sentiment1)
print("Probability:", probability1)

Sentiment: Negative
Probability: 0.007114955224096775


In [54]:
text2='I honestly did not expect much from this movie, but it completely surprised me. The storyline was beautifully written, and the performances were outstanding. Every character felt realistic, and the emotional scenes actually made me cry. The soundtrack was perfect, the cinematography was stunning, and even the ending left me smiling for hours. This is exactly the kind of movie that reminds me why I love cinema. Absolutely worth watching!'

sentiment2, probability2 = process_review(model, text2)

print("Sentiment:", sentiment2)
print("Probability:", probability2)

Sentiment: Positive
Probability: 0.9952594637870789


In [60]:
text3='The acting was incredibly convincing, especially when the characters delivered their lines like they were reading a shopping list. And that ending! Truly unforgettable, mostly because I wish I could forget the entire experience. Definitely one of the worst movies I have ever watched.'
sentiment3, probability3 = process_review(model, text3)

print("Sentiment:", sentiment3)
print("Probability:", probability3)

Sentiment: Negative
Probability: 0.2380174994468689
